# Twitter Trend Normalization (Local-Only)

Deterministic normalization of local Twitter trending artifacts for downstream exact/fuzzy/semantic matching readiness.


**Notebook purpose:** Applies deterministic normalization (NFKC, casefold, hashtag/dollar key variants) to the Twitter trending snapshot and saves normalized parquet/CSV outputs.

**Required data:** Twitter trending parquet snapshot + validation report + sample parquet + profile findings doc under `local/reference_snapshots/twitter_trending/`, `data/samples/`, and `docs/`. Produced by scripts + notebook 01.

**Run order:** Run after notebook 01 (profiling). Run before notebook 21 or 22.

## 1. Load Local Raw Snapshot


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:  # pragma: no cover
    def display(value):
        print(value)


def find_repo_root() -> Path:
    probe = Path.cwd().resolve()
    for candidate in [probe, *probe.parents]:
        if (candidate / 'src' / 'nlp' / 'trend_normalization.py').exists():
            return candidate
    raise FileNotFoundError('Could not find repository root containing src/nlp/trend_normalization.py')


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.nlp.trend_normalization import normalize_twitter_trending_dataframe

SNAPSHOT_PATH = ROOT / 'local/reference_snapshots/twitter_trending/twitter_trending_full.parquet'
SAMPLE_PATH = ROOT / 'data/samples/twitter_trending_sample_1000.parquet'
VALIDATION_REPORT_PATH = ROOT / 'local/reference_snapshots/twitter_trending/twitter_trending_validation_report.json'
PROFILE_FINDINGS_PATH = ROOT / 'docs/22_twitter_trending_profile_findings.md'

FULL_OUTPUT_PATH = ROOT / 'local/reference_snapshots/twitter_trending/twitter_trending_normalized.parquet'
SAMPLE_OUTPUT_PARQUET_PATH = ROOT / 'data/samples/twitter_trending_sample_1000_normalized.parquet'
SAMPLE_OUTPUT_CSV_PATH = ROOT / 'data/samples/twitter_trending_sample_1000_normalized.csv'

required_paths = [
    SNAPSHOT_PATH,
    SAMPLE_PATH,
    VALIDATION_REPORT_PATH,
    PROFILE_FINDINGS_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Required local normalization inputs are missing. Prior phases incomplete. Missing: '
        + ', '.join(missing_paths)
    )

validation_report = json.loads(VALIDATION_REPORT_PATH.read_text(encoding='utf-8'))
if validation_report.get('overall_pass') is not True:
    raise RuntimeError(
        'Snapshot validation report is not in PASS state. '
        f"overall_pass={validation_report.get('overall_pass')!r}"
    )
if validation_report.get('blocking_issues'):
    raise RuntimeError(
        'Snapshot validation report contains blocking issues: '
        + '; '.join(str(v) for v in validation_report['blocking_issues'])
    )

sample_df = pd.read_parquet(SAMPLE_PATH)
full_df = pd.read_parquet(SNAPSHOT_PATH)

print('repo_root:', ROOT)
print('sample_rows:', len(sample_df))
print('full_rows:', len(full_df))


repo_root: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject
sample_rows: 1000
full_rows: 101731


## 2. Load Profiling Findings Context


In [2]:
findings_preview = PROFILE_FINDINGS_PATH.read_text(encoding='utf-8').splitlines()[:40]
print('\n'.join(findings_preview))


# Twitter Trending Local Profile Findings

Date: 2026-04-03

## Dataset size
- Source (local only): `local/reference_snapshots/twitter_trending/twitter_trending_full.parquet`
- Total rows: `101,731`
- Columns: `num_hours, date, name, counts`
- Unique topics (`name`): `33,973`
- Unique dates: `749`
- Daily row-count range: `35` to `244` (median `136.0`, mean `135.82`)

## Key columns for downstream use
- `date`: day-level key for time-windowed matching.
- `name`: trend text requiring normalization.
- `counts`: heavy-tailed popularity signal (not robust for hard thresholds).
- `num_hours`: trend-duration-like signal (`1..24`) useful as secondary context.

## Duplicate findings
- Exact duplicate rows: `122`
- Duplicate `date + name` rows: `122`
- Repeated topics across dates (`count >= 2`): `14,119`
- Repeated topics with high recurrence (`count >= 10`): `1,884`
- Recommendation: deduplicate by `(date, normalized_key_no_hash)` after normalization.

## Missing-value findings
- `num_hours`:

## 3. Apply Normalization Functions To Sample Data


In [3]:
sample_normalized = normalize_twitter_trending_dataframe(sample_df)
sample_normalized.head(10)


,num_hours,date,name,counts,trend_name_raw,trend_name_clean,trend_name_clean_no_hash,trend_name_clean_no_dollar,trend_name_alnum,trend_name_token_count,trend_name_char_count,is_blank_raw,is_hashtag,has_special_chars,has_non_ascii,has_url_like,normalized_key_with_hash,normalized_key_no_hash,normalized_key_no_dollar,normalized_date
0,11,2024-03-30,Easter,1262842.0,Easter,easter,easter,easter,easter,1,6,False,False,False,False,False,easter,easter,easter,2024-03-30
1,2,2024-03-30,Good Friday,488413.0,Good Friday,good friday,good friday,good friday,good friday,2,11,False,False,False,False,False,good friday,good friday,good friday,2024-03-30
2,4,2024-03-30,#FreenBecky1stFMVN,476851.0,#FreenBecky1stFMVN,#freenbecky1stfmvn,freenbecky1stfmvn,freenbecky1stfmvn,freenbecky1stfmvn,1,17,False,True,True,False,False,#freenbecky1stfmvn,freenbecky1stfmvn,freenbecky1stfmvn,2024-03-30
3,5,2024-03-30,SAROCHA REBECCA IN VIETNAM,458880.0,SAROCHA REBECCA IN VIETNAM,sarocha rebecca in vietnam,sarocha rebecca in vietnam,sarocha rebecca in vietnam,sarocha rebecca in vietnam,4,26,False,False,False,False,False,sarocha rebecca in vietnam,sarocha rebecca in vietnam,sarocha rebecca in vietnam,2024-03-30
4,2,2024-03-30,#FHPopFestivalxNuNew,299195.0,#FHPopFestivalxNuNew,#fhpopfestivalxnunew,fhpopfestivalxnunew,fhpopfestivalxnunew,fhpopfestivalxnunew,1,19,False,True,True,False,False,#fhpopfestivalxnunew,fhpopfestivalxnunew,fhpopfestivalxnunew,2024-03-30
5,1,2024-03-30,Christian,242972.0,Christian,christian,christian,christian,christian,1,9,False,False,False,False,False,christian,christian,christian,2024-03-30
6,10,2024-03-30,Transgender Day of Visibility,241327.0,Transgender Day of Visibility,transgender day of visibility,transgender day of visibility,transgender day of visibility,transgender day of visibility,4,29,False,False,False,False,False,transgender day of visibility,transgender day of visibility,transgender day of visibility,2024-03-30
7,3,2024-03-30,wonwoo,218367.0,wonwoo,wonwoo,wonwoo,wonwoo,wonwoo,1,6,False,False,False,False,False,wonwoo,wonwoo,wonwoo,2024-03-30
8,6,2024-03-30,Christians,211825.0,Christians,christians,christians,christians,christians,1,10,False,False,False,False,False,christians,christians,christians,2024-03-30
9,3,2024-03-30,Chelsea,185466.0,Chelsea,chelsea,chelsea,chelsea,chelsea,1,7,False,False,False,False,False,chelsea,chelsea,chelsea,2024-03-30


## 4. Inspect Before/After Examples


In [4]:
comparison_cols = [
    'name',
    'trend_name_clean',
    'trend_name_clean_no_hash',
    'trend_name_clean_no_dollar',
    'trend_name_alnum',
    'is_hashtag',
    'has_special_chars',
]
sample_normalized[comparison_cols].head(25)


,name,trend_name_clean,trend_name_clean_no_hash,trend_name_clean_no_dollar,trend_name_alnum,is_hashtag,has_special_chars
0,Easter,easter,easter,easter,easter,False,False
1,Good Friday,good friday,good friday,good friday,good friday,False,False
2,#FreenBecky1stFMVN,#freenbecky1stfmvn,freenbecky1stfmvn,freenbecky1stfmvn,freenbecky1stfmvn,True,True
3,SAROCHA REBECCA IN VIETNAM,sarocha rebecca in vietnam,sarocha rebecca in vietnam,sarocha rebecca in vietnam,sarocha rebecca in vietnam,False,False
4,#FHPopFestivalxNuNew,#fhpopfestivalxnunew,fhpopfestivalxnunew,fhpopfestivalxnunew,fhpopfestivalxnunew,True,True
5,Christian,christian,christian,christian,christian,False,False
6,Transgender Day of Visibility,transgender day of visibility,transgender day of visibility,transgender day of visibility,transgender day of visibility,False,False
7,wonwoo,wonwoo,wonwoo,wonwoo,wonwoo,False,False
8,Christians,christians,christians,christians,christians,False,False
9,Chelsea,chelsea,chelsea,chelsea,chelsea,False,False


In [5]:
def sample_examples(df: pd.DataFrame, mask: pd.Series, n: int = 10) -> pd.DataFrame:
    cols = [
        'name',
        'trend_name_clean',
        'trend_name_clean_no_hash',
        'trend_name_alnum',
        'is_hashtag',
        'has_special_chars',
    ]
    return df.loc[mask, cols].drop_duplicates().head(n)

name_col = sample_normalized['name'].astype(str)
hashtag_examples = sample_examples(sample_normalized, name_col.str.strip().str.startswith('#'))
punct_examples = sample_examples(sample_normalized, name_col.str.contains(r"[^A-Za-z0-9#\s]", regex=True))
multiword_examples = sample_examples(sample_normalized, name_col.str.split().map(len) >= 2)
noisy_examples = sample_examples(sample_normalized, name_col.str.contains(r"(?:\$|\d|[^\x00-\x7F])", regex=True))

print('Hashtag examples')
display(hashtag_examples)
print('Punctuation-heavy examples')
display(punct_examples)
print('Multi-word phrase examples')
display(multiword_examples)
print('Noisy examples (ticker/digit/non-ascii)')
display(noisy_examples)


Hashtag examples


,name,trend_name_clean,trend_name_clean_no_hash,trend_name_alnum,is_hashtag,has_special_chars
2,#FreenBecky1stFMVN,#freenbecky1stfmvn,freenbecky1stfmvn,freenbecky1stfmvn,True,True
4,#FHPopFestivalxNuNew,#fhpopfestivalxnunew,fhpopfestivalxnunew,fhpopfestivalxnunew,True,True
15,#SmackDown,#smackdown,smackdown,smackdown,True,True
35,#MUFC,#mufc,mufc,mufc,True,True
44,#wellscharitycoin,#wellscharitycoin,wellscharitycoin,wellscharitycoin,True,True
48,#BREMUN,#bremun,bremun,bremun,True,True
64,#CHEBUR,#chebur,chebur,chebur,True,True
66,#DragRace,#dragrace,dragrace,dragrace,True,True
74,#murderdrones,#murderdrones,murderdrones,murderdrones,True,True
77,#DragRaceUK,#dragraceuk,dragraceuk,dragraceuk,True,True


Punctuation-heavy examples


,name,trend_name_clean,trend_name_clean_no_hash,trend_name_alnum,is_hashtag,has_special_chars
31,$MEW,$mew,$mew,mew,False,True
109,Flau’jae,flau jae,flau jae,flau jae,False,True
115,$BENJI,$benji,$benji,benji,False,True
132,$BOSI,$bosi,$bosi,bosi,False,True
139,$PLANET,$planet,$planet,planet,False,True
210,$bomeow,$bomeow,$bomeow,bomeow,False,True
211,$BOMEOW,$bomeow,$bomeow,bomeow,False,True
216,$SSNC,$ssnc,$ssnc,ssnc,False,True
230,Martin Luther King Jr.,martin luther king jr,martin luther king jr,martin luther king jr,False,True
246,dr. king,dr king,dr king,dr king,False,True


Multi-word phrase examples


,name,trend_name_clean,trend_name_clean_no_hash,trend_name_alnum,is_hashtag,has_special_chars
1,Good Friday,good friday,good friday,good friday,False,False
3,SAROCHA REBECCA IN VIETNAM,sarocha rebecca in vietnam,sarocha rebecca in vietnam,sarocha rebecca in vietnam,False,False
6,Transgender Day of Visibility,transgender day of visibility,transgender day of visibility,transgender day of visibility,False,False
12,Happy Easter,happy easter,happy easter,happy easter,False,False
20,Secret Service,secret service,secret service,secret service,False,False
36,NC State,nc state,nc state,nc state,False,False
38,West Ham,west ham,west ham,west ham,False,False
39,Chance Perdomo,chance perdomo,chance perdomo,chance perdomo,False,False
49,Good Saturday,good saturday,good saturday,good saturday,False,False
53,Big 4,big 4,big 4,big 4,False,False


Noisy examples (ticker/digit/non-ascii)


,name,trend_name_clean,trend_name_clean_no_hash,trend_name_alnum,is_hashtag,has_special_chars
2,#FreenBecky1stFMVN,#freenbecky1stfmvn,freenbecky1stfmvn,freenbecky1stfmvn,True,True
31,$MEW,$mew,$mew,mew,False,True
53,Big 4,big 4,big 4,big 4,False,False
60,Elite 8,elite 8,elite 8,elite 8,False,False
62,ELITE 8,elite 8,elite 8,elite 8,False,False
94,#80million,#80million,80million,80million,True,True
109,Flau’jae,flau jae,flau jae,flau jae,False,True
115,$BENJI,$benji,$benji,benji,False,True
132,$BOSI,$bosi,$bosi,bosi,False,True
139,$PLANET,$planet,$planet,planet,False,True


## 5. Validate Helper Columns


In [6]:
helper_columns = [
    'trend_name_raw',
    'trend_name_clean',
    'trend_name_clean_no_hash',
    'trend_name_clean_no_dollar',
    'trend_name_alnum',
    'trend_name_token_count',
    'trend_name_char_count',
    'is_hashtag',
    'has_special_chars',
    'has_non_ascii',
    'normalized_key_with_hash',
    'normalized_key_no_hash',
    'normalized_key_no_dollar',
    'normalized_date',
]

missing_helper_columns = [c for c in helper_columns if c not in sample_normalized.columns]
if missing_helper_columns:
    raise AssertionError('Missing helper columns: ' + ', '.join(missing_helper_columns))

sample_normalized[helper_columns].head(10)


,trend_name_raw,trend_name_clean,trend_name_clean_no_hash,trend_name_clean_no_dollar,trend_name_alnum,trend_name_token_count,trend_name_char_count,is_hashtag,has_special_chars,has_non_ascii,normalized_key_with_hash,normalized_key_no_hash,normalized_key_no_dollar,normalized_date
0,Easter,easter,easter,easter,easter,1,6,False,False,False,easter,easter,easter,2024-03-30
1,Good Friday,good friday,good friday,good friday,good friday,2,11,False,False,False,good friday,good friday,good friday,2024-03-30
2,#FreenBecky1stFMVN,#freenbecky1stfmvn,freenbecky1stfmvn,freenbecky1stfmvn,freenbecky1stfmvn,1,17,True,True,False,#freenbecky1stfmvn,freenbecky1stfmvn,freenbecky1stfmvn,2024-03-30
3,SAROCHA REBECCA IN VIETNAM,sarocha rebecca in vietnam,sarocha rebecca in vietnam,sarocha rebecca in vietnam,sarocha rebecca in vietnam,4,26,False,False,False,sarocha rebecca in vietnam,sarocha rebecca in vietnam,sarocha rebecca in vietnam,2024-03-30
4,#FHPopFestivalxNuNew,#fhpopfestivalxnunew,fhpopfestivalxnunew,fhpopfestivalxnunew,fhpopfestivalxnunew,1,19,True,True,False,#fhpopfestivalxnunew,fhpopfestivalxnunew,fhpopfestivalxnunew,2024-03-30
5,Christian,christian,christian,christian,christian,1,9,False,False,False,christian,christian,christian,2024-03-30
6,Transgender Day of Visibility,transgender day of visibility,transgender day of visibility,transgender day of visibility,transgender day of visibility,4,29,False,False,False,transgender day of visibility,transgender day of visibility,transgender day of visibility,2024-03-30
7,wonwoo,wonwoo,wonwoo,wonwoo,wonwoo,1,6,False,False,False,wonwoo,wonwoo,wonwoo,2024-03-30
8,Christians,christians,christians,christians,christians,1,10,False,False,False,christians,christians,christians,2024-03-30
9,Chelsea,chelsea,chelsea,chelsea,chelsea,1,7,False,False,False,chelsea,chelsea,chelsea,2024-03-30


In [7]:
helper_quality = {
    'sample_rows': int(len(sample_normalized)),
    'sample_unique_raw_name': int(sample_normalized['name'].nunique(dropna=True)),
    'sample_unique_normalized_no_hash': int(sample_normalized['normalized_key_no_hash'].nunique(dropna=True)),
    'sample_duplicates_date_normalized_no_hash': int(
        sample_normalized.duplicated(subset=['date', 'normalized_key_no_hash']).sum()
    ),
    'sample_hashtag_rows': int(sample_normalized['is_hashtag'].sum()),
    'sample_special_char_rows': int(sample_normalized['has_special_chars'].sum()),
}
helper_quality


{'sample_rows': 1000,
 'sample_unique_raw_name': 909,
 'sample_unique_normalized_no_hash': 884,
 'sample_duplicates_date_normalized_no_hash': 22,
 'sample_hashtag_rows': 187,
 'sample_special_char_rows': 243}

## 6. Apply Normalization To Full Local Snapshot


In [8]:
full_normalized = normalize_twitter_trending_dataframe(full_df)

full_quality = {
    'full_rows': int(len(full_normalized)),
    'full_unique_raw_name': int(full_normalized['name'].nunique(dropna=True)),
    'full_unique_normalized_no_hash': int(full_normalized['normalized_key_no_hash'].nunique(dropna=True)),
    'full_duplicates_date_normalized_no_hash': int(
        full_normalized.duplicated(subset=['date', 'normalized_key_no_hash']).sum()
    ),
    'full_hashtag_rows': int(full_normalized['is_hashtag'].sum()),
    'full_special_char_rows': int(full_normalized['has_special_chars'].sum()),
}
full_quality


{'full_rows': 101731,
 'full_unique_raw_name': 33973,
 'full_unique_normalized_no_hash': 32208,
 'full_duplicates_date_normalized_no_hash': 1892,
 'full_hashtag_rows': 17180,
 'full_special_char_rows': 20818}

## 7. Save Normalized Outputs


In [9]:
FULL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
SAMPLE_OUTPUT_PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)

full_normalized.to_parquet(FULL_OUTPUT_PATH, index=False)
sample_normalized.to_parquet(SAMPLE_OUTPUT_PARQUET_PATH, index=False)
sample_normalized.to_csv(SAMPLE_OUTPUT_CSV_PATH, index=False)

print('wrote:', FULL_OUTPUT_PATH)
print('wrote:', SAMPLE_OUTPUT_PARQUET_PATH)
print('wrote:', SAMPLE_OUTPUT_CSV_PATH)


wrote: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/reference_snapshots/twitter_trending/twitter_trending_normalized.parquet
wrote: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/data/samples/twitter_trending_sample_1000_normalized.parquet
wrote: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/data/samples/twitter_trending_sample_1000_normalized.csv


## 8. Summary Recommendation For Next Step


In [10]:
exact_match_ready = True
fuzzy_candidate_ready = True
semantic_fallback_ready = True

print('exact_match_ready:', exact_match_ready)
print('fuzzy_candidate_ready:', fuzzy_candidate_ready)
print('semantic_fallback_ready:', semantic_fallback_ready)
print('next_phase: local Bluesky text preparation + exact-first matching pipeline with fuzzy/semantic fallback tiers')


exact_match_ready: True
fuzzy_candidate_ready: True
semantic_fallback_ready: True
next_phase: local Bluesky text preparation + exact-first matching pipeline with fuzzy/semantic fallback tiers
